제 1유형

1. ProductA 가격과 ProductB 가격이 모두 0원이 아닌 데이터를 필터링하고, ProductA 와 ProductB의 가격 차이를 정의하시오. 각 도시별 가격 차이의 평균 중 가장 큰 값을 구하시오.(소수점 첫째 자리까지 반올림)

2. 100명의 키와 몸무게를 조사하여 적정 체중인지 판단할 수 있는 BMI를 산출하려 한다. 아래 표를 참고하여 BMI를 기준으로 저체중, 정상, 과체중, 비만을 구분하고, 저체중인 사람과 비만인 사람의 총합을 구하시오.

BMI 기준
저체중: 18.5 미만
정상: 18.5 이상 23 미만
과체중: 23 이상 25 미만
비만: 25 이상

BMI = 몸무게(kg) / 키 제곱
(소수점 첫째 자리까지 반올림)

3. 연도별로 가장 큰 순생산량(생산된 제품 수 - 판매된 제품 수)을 가진 공장을 찾고, 순생산량의 합을 구하시오.

In [42]:
#1. ProductA 가격과 ProductB 가격이 모두 0원이 아닌 데이터를 필터링하고, ProductA 와 ProductB의 가격 차이를 정의하시오. 각 도시별 가격 차이의 평균 중 가장 큰 값을 구하시오.(소수점 첫째 자리까지 반올림)
import pandas as pd
df = pd.read_csv('6_1_1.csv')
#print(df.head())

q1 = df[(df['ProductA가격'] != 0) & (df['ProductB가격'] != 0)].copy()
#print(q1.head())
import numpy as np
q1['diff'] = np.abs(q1['ProductA가격'] - q1['ProductB가격'])
q2 = q1.groupby('도시명')['diff'].mean().sort_values(ascending=False)
print(q2.index[0])
print(round(q2.iloc[0], 1))

대전
16333.3


2. 100명의 키와 몸무게를 조사하여 적정 체중인지 판단할 수 있는 BMI를 산출하려 한다. 아래 표를 참고하여 BMI를 기준으로 저체중, 정상, 과체중, 비만을 구분하고, 저체중인 사람과 비만인 사람의 총합을 구하시오.

BMI 기준
저체중: 18.5 미만
정상: 18.5 이상 23 미만
과체중: 23 이상 25 미만
비만: 25 이상

BMI = 몸무게(kg) / 키 제곱
(소수점 첫째 자리까지 반올림)

In [17]:
import pandas as pd
df = pd.read_csv('6_1_2.csv')
#print(df.head())

df['Height_m'] = df['Height_cm'] / 100
df['BMI'] = df['Weight_kg'] / (df['Height_m'] * df['Height_m'])
df['BMI'] = df['BMI'].round(1)

def categorize(bmi):
    if bmi < 18.5: return '저체중'
    elif 18.5 <= bmi < 23: return '정상'
    elif 23 <= bmi < 25: return '과체중'
    else: return '비만'

df['class'] = df['BMI'].apply(categorize)
print(df.head())

class_lower = len(df[df['class']=='저체중'])
class_over = len(df[df['class']=='비만'])

print(class_lower + class_over)

    Height_cm   Weight_kg  Height_m   BMI class
0  165.021320   66.131592  1.650213  24.3   과체중
1  183.219470   82.164648  1.832195  24.5   과체중
2  140.006862  110.875368  1.400069  56.6    비만
3  158.139954   68.581581  1.581400  27.4    비만
4  148.805353  112.682812  1.488054  50.9    비만
74


In [30]:
#3. 연도별로 가장 큰 순생산량(생산된 제품 수 - 판매된 제품 수)을 가진 공장을 찾고, 순생산량의 합을 구하시오.
import pandas as pd
df = pd.read_csv('6_1_3.csv')
#print(df.head())

df['made'] = df['products_made_domestic'] + df['products_made_international']
df['sold'] = df['products_sold_domestic'] + df['products_sold_international']
df['original'] = df['made'] - df['sold']

idx = df.groupby('year')['original'].idxmax()
result_df = df.loc[idx]
print(result_df[['year', 'factory', 'original']])


    year    factory  original
12  2020  Factory E      2034
75  2021  Factory A      1248
96  2022  Factory E      1474
49  2023  Factory B      1032


제 2유형

훈련 데이터로 학습한 모델을 테스트 데이터에 적용하여 예측한 결과를 제출하시오.(Target: DBP)

% 제출 형식은 ID, pred 두 칼럼만 존재해야 한다.(평가 지표: RMSE)

In [21]:
import pandas as pd
train = pd.read_csv('6_2_train.csv')
test = pd.read_csv('6_2_test.csv')

#print(train.info())
# 결측치, 범주형: Gender

train['Gender'].fillna(train['Gender'].mode()[0], inplace=True)
test['Gender'].fillna(train['Gender'].mode()[0], inplace=True)

X = train.drop(['ID', 'DBP'], axis=1)
y = train['DBP']
X_submit = test.drop(['ID', 'DBP'], axis=1)

X = pd.get_dummies(X, columns=['Gender'])
X_submit = pd.get_dummies(X_submit, columns=['Gender'])

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_val)

from sklearn.metrics import mean_squared_error
import numpy as np
score = np.sqrt(mean_squared_error(y_val, pred))
print(score)

model = RandomForestRegressor(random_state=42)
model.fit(X, y)
final_pred = model.predict(X_submit)

result = pd.DataFrame({
    'ID': test['ID'],
    'pred': final_pred
})
print(result.head())

C:\Users\sangh\AppData\Local\Temp\ipykernel_3116\721740840.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['Gender'].fillna(train['Gender'].mode()[0], inplace=True)
C:\Users\sangh\AppData\Local\Temp\ipykernel_3116\721740840.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a cop

11.74089608946466
     ID   pred
0  2856  76.12
1  3116  87.18
2  1125  76.09
3  3604  82.04
4  1334  77.15


제 3유형

<3.1. 업무 수행 시간 조사, K-S 검정을 통해 업무 수행 시간이 정규분포 따르는지 검정>

1. 직원들의 업무 수행 시간의 평균과 표준편차를 구하시오.(소수점 셋째 자리까지 반올림)

2. 직원들의 업무 수행 시간이 정규분포를 따르는 지 K-S 검정을 실시하고, 검정통계량을 계산하시오.(소수점 첫째 자리까지 반올림)

3. p-value를 바탕으로 유의수준 5%에서 귀무가설의 기각/채택 여부를 결정하시오.(p-value는 소수점 셋째 자리까지 반올림)

<3.2. 주택들의 가격, 면적, 방의 개수, 연식을 조사>

2.1. 주택 가격을 종속 변수로 하고, 면적, 방의 개수, 연식을 독립 변수로 하는 다중회귀 분석을 수행하여, 회귀 계수가 가장 높은 변수를 구하시오.

2.2. 유의수준 5% 하에서 각 독립 변수가 주택 가격에 미치는 영향이 통계적으로 유의미한지 판단하고, 유의미한 변수 개수를 구하시오.

In [26]:
#<3.1. 업무 수행 시간 조사, K-S 검정을 통해 업무 수행 시간이 정규분포 따르는지 검정>
#1. 직원들의 업무 수행 시간의 평균과 표준편차를 구하시오.(소수점 셋째 자리까지 반올림)
#2. 직원들의 업무 수행 시간이 정규분포를 따르는 지 K-S 검정을 실시하고, 검정통계량을 계산하시오.(소수점 첫째 자리까지 반올림)
#3. p-value를 바탕으로 유의수준 5%에서 귀무가설의 기각/채택 여부를 결정하시오.(p-value는 소수점 셋째 자리까지 반올림)
import pandas as pd
df = pd.read_csv('6_3_1.csv')
#print(df.head())

from scipy.stats import kstest
mean = df['work_hours'].mean()
std = df['work_hours'].std()
print(round(mean, 3))
print(round(std, 3))
data_scaled = (df['work_hours'] - mean) / std
stat, p_val = kstest(data_scaled, 'norm')
print(round(stat, 1))
print(round(p_val, 3))
# 채택

8.09
1.519
0.1
0.778


In [36]:
# <3.2. 주택들의 가격, 면적, 방의 개수, 연식을 조사>
#2.1. 주택 가격을 종속 변수로 하고, 면적, 방의 개수, 연식을 독립 변수로 하는 다중회귀 분석을 수행하여, 회귀 계수가 가장 높은 변수를 구하시오.
#2.2. 유의수준 5% 하에서 각 독립 변수가 주택 가격에 미치는 영향이 통계적으로 유의미한지 판단하고, 유의미한 변수 개수를 구하시오.
import pandas as pd
df = pd.read_csv('6_3_2.csv')
#print(df.head())

from statsmodels.formula.api import ols
model = ols('price ~ area + rooms + age', data=df).fit()
coef = model.params
#print(coef)
# 답: rooms

#print(round(model.pvalues, 4))
# 답: 3개
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.708
Model:                            OLS   Adj. R-squared:                  0.699
Method:                 Least Squares   F-statistic:                     77.65
Date:                Wed, 26 Nov 2025   Prob (F-statistic):           1.41e-25
Time:                        16:43:59   Log-Likelihood:                -1055.4
No. Observations:                 100   AIC:                             2119.
Df Residuals:                      96   BIC:                             2129.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   5.395e+04   6081.703      8.871      0.0